<a id="intro"></a>
# ЛР № 5. Расчёт метрик Green Taxi

Это первая часть общего аналитического ноутбука ЛР № 5 и № 7.
Здесь читаем готовый `mart`, рассчитываем метрики и проверяем результаты.
В седьмой работе ниже добавляются
визуализации и исследование. Промежуточные файлы данных не создаются.

Эти Markdown-ячейки служат учебными подсказками. При выполнении работы их текст дублировать не требуется. Все обязательные описания, определения и вывод собраны в
[docs/metrics.md](../docs/metrics.md). В сдаваемом ноутбуке нужны работающий код, SQL, результаты расчётов и проверок.

Предыдущий этап: [витрина ЛР № 4](../docs/mart.md).
Сквозной вопрос: [Project Brief](../docs/project_brief.md).


## 1. Вход и параметры

Используем проектное ядро `.venv` и существующий S3-стенд.
Повторный запуск всей первой части восстанавливает результаты после перезапуска ядра.


In [1]:
# M01. Параметры
DATASET_SLUG = "green_tripdata"
SOURCE_INGESTED_ON = "2026-09-30"
MART_PATH = f"s3://mart/{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}/trips.parquet"
PERIOD_START = "2025-01-01"
PERIOD_END = "2025-02-01"
print(f"Вход: {MART_PATH}")
print(f"Период по pickup_at: [{PERIOD_START}, {PERIOD_END})")


Вход: s3://mart/green_tripdata/source_ingested_on=2026-09-30/trips.parquet
Период по pickup_at: [2025-01-01, 2025-02-01)


## 2. Подключение

Переиспользуем подключение из предыдущих работ. Настройки читаются из `.env`. Вспомогательная функция только выполняет SQL
и выводит таблицу. Вся логика расчётов расположена в явных запросах ниже.


In [2]:
# M02. Подключение
import os
import sys
from html import escape
from pathlib import Path

import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in settings if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing)}")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402

connection = duckdb.connect()
configure_s3(connection)


def literal(value):
    return "'" + value.replace("'", "''") + "'"


def select_table(sql):
    cursor = connection.execute(sql)
    description = cursor.description
    if description is None:
        raise ValueError("Ожидается таблица результата")
    rows = cursor.fetchall()
    header = "".join(f"<th>{escape(column[0])}</th>" for column in description)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    print(f"Строк результата: {len(rows)}")
    return rows


## 3. Чтение витрины

Одна строка `trips` - сохранённая запись о поездке. Надёжного бизнес-ключа в исходном датасете нет.
Общий фильтр задаёт только отчётный период. Отрицательные суммы пока не исключаются.
`analysis_base` один раз загружает выбранные данные из S3 в память DuckDB.
Следующие расчёты и будущие графики используют данные в памяти этого подключения.


In [3]:
# M03. Вход
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW trips AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
required_fields = {
    "pickup_at", "pickup_borough", "pickup_hour", "pickup_isodow",
    "duration_minutes", "total_amount", "negative_total_flag",
}
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE trips").fetchall()}
assert not (required_fields - schema.keys()), "В витрине отсутствуют нужные поля"
connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE analysis_base AS
    SELECT * FROM trips
    WHERE pickup_at >= TIMESTAMP {literal(PERIOD_START)}
      AND pickup_at < TIMESTAMP {literal(PERIOD_END)}
""")
input_rows = select_table("""
                          SELECT count(*)       AS records_count,
                                 count(*)          FILTER (WHERE total_amount < 0) AS negative_count, count(*) FILTER (WHERE negative_total_flag) AS flagged_count, count(total_amount) FILTER (WHERE total_amount >= 0) AS cost_count, min(pickup_at) AS first_pickup,
                                 max(pickup_at) AS last_pickup
                          FROM analysis_base
                          """)
print(f"Физических полей витрины: {len(schema)}")


records_count,negative_count,flagged_count,cost_count,first_pickup,last_pickup
48235,152,152,48083,2025-01-01 00:01:11,2025-01-31 23:59:44


Строк результата: 1
Физических полей витрины: 15


<a id="definitions"></a>
## 4. Что считаем

Три метрики:
- число записей,
- медианная длительность в минутах,
- средняя стоимость в USD.

Средння стоимость рассчитывается только по заполненным неотрицательным суммам. Ноль входит в расчёт средней стоимости. Запись с отрицательной суммой остаётся для подсчёта количества поездок и длительности.

Полные определения и пояснение окна находятся в `docs/metrics.md`.


## 5. Общий итог

Фильтр стоимости находится у конкретных агрегатов, поскольку общий `WHERE` на сумму
изменил бы одновременно и количество и медиану.

In [4]:
# M04. Общие метрики
connection.execute("""
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT count(*) AS records_count,
        count(total_amount) FILTER (WHERE total_amount >= 0) AS cost_count,
        median(duration_minutes) AS median_minutes,
        avg(total_amount) FILTER (WHERE total_amount >= 0) AS avg_amount
    FROM analysis_base
""")
overall_rows = select_table("""
                            SELECT records_count,
                                   cost_count,
                                   round(median_minutes, 2) AS median_minutes,
                                   round(avg_amount, 2)     AS avg_amount
                            FROM overall_metrics
                            """)

overall_raw = connection.execute("SELECT * FROM overall_metrics").fetchone()
assert overall_raw is not None


records_count,cost_count,median_minutes,avg_amount
48235,48083,11.27,22.75


Строк результата: 1


## 6. Сравнение по часам и окно

Одна строка результата - час суток за все дни и территории выбранного месяца.
Окно добавляет долю группы в общем количестве, сохраняя 24 часовые строки.


In [5]:
# M05. По часам
connection.execute("""
    CREATE OR REPLACE TEMP TABLE hourly_metrics AS
    WITH grouped AS (
        SELECT pickup_hour,
            count(*) AS records_count,
            count(total_amount) FILTER (WHERE total_amount >= 0) AS cost_count,
            sum(total_amount) FILTER (WHERE total_amount >= 0) AS cost_sum,
            median(duration_minutes) AS median_minutes,
            avg(total_amount) FILTER (WHERE total_amount >= 0) AS avg_amount
        FROM analysis_base
        GROUP BY pickup_hour
    )
    SELECT *,
           100.0 * records_count
              / nullif(sum(records_count) OVER (), 0) AS percent_of_records
    FROM grouped
""")
hourly_rows = select_table("""
                           SELECT pickup_hour,
                                  records_count,
                                  cost_count,
                                  round(median_minutes, 2)     AS median_minutes,
                                  round(avg_amount, 2)         AS avg_amount,
                                  round(percent_of_records, 3) AS percent_of_records
                           FROM hourly_metrics
                           ORDER BY pickup_hour
                           """)
hourly_raw = connection.execute("""
                                SELECT *
                                FROM hourly_metrics
                                ORDER BY pickup_hour
                                """).fetchall()


pickup_hour,records_count,cost_count,median_minutes,avg_amount,percent_of_records
0,716,715,9.15,24.43,1.484
1,538,536,8.35,25.87,1.115
2,367,363,8.9,26.29,0.761
3,273,269,8.8,26.0,0.566
4,240,237,9.69,28.89,0.498
5,274,272,10.2,27.34,0.568
6,890,889,9.17,21.28,1.845
7,1916,1913,11.23,21.84,3.972
8,2418,2416,12.23,22.39,5.013
9,2532,2525,12.04,23.44,5.249


Строк результата: 24


## 7. Уточнение и отдельные значения

Добавляем район посадки к группировке, сохраняя определения метрик.
Временная таблица содержит все имеющиеся сочетания района и часа.
Отсутствующие сочетания не подменяются нулём.
Для просмотра ниже показаны первые 10 строк; полный результат остаётся в памяти.

Отдельная выборка длительностей нужна для будущего распределения. Она сохраняет
все записи, включая отрицательные суммы. Восстановить её из медиан часов нельзя.


In [6]:
# M06. Результаты для продолжения анализа
connection.execute("""
    CREATE OR REPLACE TEMP TABLE segment_metrics AS
    SELECT pickup_borough, pickup_hour,
        count(*) AS records_count,
        count(total_amount) FILTER (WHERE total_amount >= 0) AS cost_count,
        sum(total_amount) FILTER (WHERE total_amount >= 0) AS cost_sum,
        median(duration_minutes) AS median_minutes,
        avg(total_amount) FILTER (WHERE total_amount >= 0) AS avg_amount
    FROM analysis_base GROUP BY pickup_borough, pickup_hour
""")
print("Первые 10 строк группировки по району и часу:")
segment_preview = select_table("""
    SELECT pickup_borough, pickup_hour, records_count, cost_count,
        round(median_minutes, 2) AS median_minutes,
        round(avg_amount, 2) AS avg_amount
    FROM segment_metrics
    ORDER BY pickup_borough, pickup_hour
    LIMIT 10
""")
segment_totals = select_table("""
                              SELECT pickup_borough,
                                     sum(records_count) AS records_count,
                                     sum(cost_count)    AS cost_count
                              FROM segment_metrics
                              GROUP BY pickup_borough
                              ORDER BY records_count DESC, pickup_borough
                              """)
segment_rows = connection.execute("""
                                  SELECT *
                                  FROM segment_metrics
                                  ORDER BY pickup_borough, pickup_hour
                                  """).fetchall()
print(f"Строк территориальной группировки: {len(segment_rows)}")
connection.execute("""
    CREATE OR REPLACE TEMP TABLE distribution_values AS
    SELECT duration_minutes FROM analysis_base
""")
duration_summary = select_table("""
                                SELECT count(*)                 AS records_count,
                                       min(duration_minutes)    AS min_minutes,
                                       median(duration_minutes) AS median_minutes,
                                       max(duration_minutes)    AS max_minutes
                                FROM distribution_values
                                """)[0]
durations = [row[0] for row in connection.execute("""
                                                  SELECT duration_minutes
                                                  FROM distribution_values
                                                  ORDER BY duration_minutes
                                                  """).fetchall()]


Первые 10 строк группировки по району и часу:


pickup_borough,pickup_hour,records_count,cost_count,median_minutes,avg_amount
Bronx,0,17,17,5.72,20.91
Bronx,1,7,7,6.55,19.5
Bronx,2,12,11,6.24,15.86
Bronx,3,5,5,7.0,17.4
Bronx,4,16,16,10.15,21.72
Bronx,5,18,16,11.2,22.24
Bronx,6,20,20,26.38,40.87
Bronx,7,12,12,21.13,31.54
Bronx,8,40,40,17.89,25.02
Bronx,9,66,66,21.27,31.74


Строк результата: 10


pickup_borough,records_count,cost_count
Manhattan,30059,29989
Queens,10710,10651
Brooklyn,6782,6763
Bronx,566,562
Unknown,97,97
N/A,18,18
Staten Island,3,3


Строк результата: 7
Строк территориальной группировки: 134


records_count,min_minutes,median_minutes,max_minutes
48235,0.016666666666666666,11.266666666666667,1439.65


Строк результата: 1


<a id="validation"></a>
## 8. Проверка расчётов

Проверяем количество, число подходящих сумм и правильное объединение средних.
Для денежного среднего используем веса по `cost_count`, а не по числу всех строк.
Числа с плавающей точкой сравниваются с небольшим допуском. Краткие объяснения
ожидаемых и фактических результатов записаны в `docs/metrics.md`.


In [21]:
# M07. Проверки
check_rows = select_table("""
                          WITH checks AS (SELECT 1                           AS check_no,
                                                 'Сумма количества по часам' AS check_name,
                                                 (SELECT sum(records_count) FROM hourly_metrics)::DOUBLE AS actual, (SELECT records_count FROM overall_metrics) ::DOUBLE AS expected
                                          UNION ALL
                                          SELECT 2,
                                                 'Сумма количества по районам и часам',
                                                 (SELECT sum(records_count) FROM segment_metrics),
                                                 (SELECT records_count FROM overall_metrics)
                                          UNION ALL
                                          SELECT 3,
                                                 'Количество сумм для средней стоимости',
                                                 (SELECT sum(cost_count) FROM segment_metrics),
                                                 (SELECT count(total_amount) FROM analysis_base WHERE total_amount >= 0)
                                          UNION ALL
                                          SELECT 4,
                                                 'Средняя стоимость совпадает при расчёте по записям и по часовым итогам',
                                                 (SELECT sum(cost_sum) / nullif(sum(cost_count), 0)
                                                  FROM hourly_metrics),
                                                 (SELECT avg_amount FROM overall_metrics))
                          SELECT check_name,
                                 round(actual, 2)              AS actual,
                                 round(expected, 2)            AS expected,
                                 abs(actual - expected) < 1e-4 AS passed
                          FROM checks
                          ORDER BY check_no
                          """)
assert all(row[3] for row in check_rows), "Проверки не сошлись: вернитесь к расчётам"
print(f"Проверок пройдено: {len(check_rows)} из {len(check_rows)}")


check_name,actual,expected,passed
Сумма количества по часам,48235.0,48235.0,True
Сумма количества по районам и часам,48235.0,48235.0,True
Количество сумм для средней стоимости,48083.0,48083.0,True
Средняя стоимость совпадает при расчёте по записям и по часовым итогам,22.75,22.75,True


Строк результата: 4
Проверок пройдено: 4 из 4


<a id="continuation"></a>
## Продолжение в ЛР № 7

Первая часть завершена. Пояснения и численный вывод находятся в
[docs/metrics.md](../docs/metrics.md). Подготовлены временные таблицы `hourly_metrics`,
`segment_metrics`, `distribution_values`; общий итог - в `overall_metrics`.
Для работы из Python доступны `overall_raw`, `hourly_raw`, `segment_rows`,
`segment_totals`, `durations` и `duration_summary`. Эти данные не округлены.
`hourly_rows` и `overall_rows` служат для чтения округлённого вывода.

В седьмой работе ниже будут добавлены ячейки визуализации и исследования.
После перезапуска ядра сначала выполните M01-M07: все результаты должны
восстанавливаться из кода, а не зависеть от случайно сохранившейся памяти.
Изменив условия в расчётном блоке, повторите зависимые ячейки и проверки,
затем обновите описания в `metrics.md` и последующие графики.
